In [1]:
#had issue where i had planned to scrape the not included bpm (like tunebat) from song bpm databases but none of my code could read any of the websites so i refined this one to have a higher match rate.

import subprocess
subprocess.run(['pip', 'install', 'thefuzz', 'python-Levenshtein', '-q'])
print('ready')

ready


In [3]:
import pandas as pd
import numpy as np
from thefuzz import fuzz
import os
import re
import kagglehub

print('import successful')

imports successful


In [5]:
#loading billboard data
url = 'https://raw.githubusercontent.com/Liviah27/Empirical-Project-Data-Science-26/refs/heads/raw/billboard_data.csv'
df_billboard = pd.read_csv(url)

print(f'Billboard songs loaded: {len(df_billboard):,}')
print(df_billboard.head())


Billboard songs loaded: 5,100
   year  position                           title              artist
0  1975         1      Love Will Keep Us Together  Captain & Tennille
1  1975         2               Rhinestone Cowboy       Glen Campbell
2  1975         3            Philadelphia Freedom          Elton John
3  1975         4  Before the Next Teardrop Falls       Freddy Fender
4  1975         5              My Eyes Adored You       Frankie Valli


In [7]:
#loading kaggle dataset making sure to combine both csv files from the dataset: Nov2018 and April2019
path = kagglehub.dataset_download('tomigelo/spotify-audio-features')

df_nov = pd.read_csv(os.path.join(path, 'SpotifyAudioFeaturesNov2018.csv'))
df_apr = pd.read_csv(os.path.join(path, 'SpotifyAudioFeaturesApril2019.csv'))

#stacking both files and drop duplicate track_ids
df_kaggle = pd.concat([df_nov, df_apr], ignore_index=True)
df_kaggle = df_kaggle.drop_duplicates(subset='track_id')

print(f'Combined Kaggle tracks: {len(df_kaggle):,}')
print(f'Nov file: {len(df_nov):,} | Apr file: {len(df_apr):,}')

Combined Kaggle tracks: 130,989
Nov file: 116,372  |  Apr file: 130,663


In [9]:
def clean_string(s):
    if not isinstance(s, str):
        return ''
    s = s.lower()
    # removing featured artist text
    #this was the main reason kaggle track names failed to match billboard titles before
    s = re.sub(r'\bfeat\.?\b.*', '', s)
    s = re.sub(r'\bft\.?\b.*',   '', s)
    s = re.sub(r'\bwith\b.*',    '', s)
    #removing anything in brackets
    s = re.sub(r'[\(\[\{].*?[\)\]\}]', '', s)
    #removing punctuation
    s = re.sub(r"[^a-z0-9\s]", '', s)
    #collapsing any whitespace
    s = re.sub(r'\s+', ' ', s).strip()
    return s

#cleaning title and artist separately 
df_kaggle['clean_title'] = df_kaggle['track_name'].apply(clean_string)
df_kaggle['clean_artist'] = df_kaggle['artist_name'].apply(clean_string)

df_billboard['clean_title'] = df_billboard['title'].apply(clean_string)
df_billboard['clean_artist'] = df_billboard['artist'].apply(clean_string)

#building a lookup dictionary grouping by title helps quickly find all songs with a matching name without scanning all rows for every song
kaggle_by_title = df_kaggle.groupby('clean_title')

print('Cleaning done.')
print('Sample Billboard clean titles:', df_billboard['clean_title'].head(5).tolist())
print('Sample Kaggle clean titles:', df_kaggle['clean_title'].head(5).tolist())

Cleaning done.
Sample Billboard clean titles: ['love will keep us together', 'rhinestone cowboy', 'philadelphia freedom', 'before the next teardrop falls', 'my eyes adored you']
Sample Kaggle clean titles:    ['big bank', 'band drum', 'radio silence', 'lactose', 'same original mix']


In [11]:
#finding kaggle songs where the title matches the Billboard title using token_set_ratio
#then among title matches it picks the one whose artist best matches preventing wrong songs with the same title being returned

#prebuilding a list of all unique clean titles in kaggle for fast title search
kaggle_titles = df_kaggle['clean_title'].unique().tolist()

TITLE_THRESHOLD  = 90   #how similar the title must be 
ARTIST_THRESHOLD = 70   #looser as artist names vary more

def find_match(billboard_title, billboard_artist):
    #finding Kaggle titles that closely match the Billboard title
    # token_set_ratio handles the case where one string contains the other
    title_candidates = [
        t for t in kaggle_titles
        if fuzz.token_set_ratio(billboard_title, t) >= TITLE_THRESHOLD
    ]

    if not title_candidates:
        return np.nan, np.nan

    #among title matches, find the best artist match
    best_score = -1
    best_row   = None

    for title in title_candidates:
        try:
            candidates_df = kaggle_by_title.get_group(title)
        except KeyError:
            continue

        for _, krow in candidates_df.iterrows():
            artist_score = fuzz.token_set_ratio(billboard_artist, krow['clean_artist'])
            if artist_score >= ARTIST_THRESHOLD and artist_score > best_score:
                best_score = artist_score
                best_row   = krow

    if best_row is not None:
        return best_row['tempo'], best_row['valence']

    return np.nan, np.nan

print('Matching function defined.')

Matching function defined.


In [13]:
#test a handful of songs to confirm the new matcher is working 

test_cases = [
    ('billie jean', 'michael jackson'),
    ('bohemian rhapsody', 'queen'),
    ('before the next teardrop falls', 'freddy fender'),
    ('shining star', 'earth wind fire'),
    ('shape of you', 'ed sheeran'),
]

for title, artist in test_cases:
    tempo, valence = find_match(title, artist)
    print(f'{artist} - {title}: BPM={tempo}, valence={valence}')

michael jackson - billie jean: BPM=138.121, valence=0.708
queen - bohemian rhapsody: BPM=144.031, valence=0.246
freddy fender - before the next teardrop falls: BPM=91.041, valence=0.722
earth wind fire - shining star: BPM=102.587, valence=0.847
ed sheeran - shape of you: BPM=nan, valence=nan


In [15]:
# matching loop
bpm_values = []
valence_values = []

for i, row in df_billboard.iterrows():
    tempo, valence = find_match(row['clean_title'], row['clean_artist'])
    bpm_values.append(tempo)
    valence_values.append(valence)

    if (i + 1) % 500 == 0:
        found = sum(1 for v in bpm_values if not (isinstance(v, float) and np.isnan(v)))
        print(f'Progress: {i+1}/{len(df_billboard)} | Matched: {found} ({found/(i+1):.1%})')

df_billboard['bpm'] = bpm_values
df_billboard['valence'] = valence_values

matched = df_billboard['bpm'].notna().sum()
total = len(df_billboard)
print(f'\nFinal match rate: {matched}/{total} ({matched/total:.1%})')

Progress: 500/5100 | Matched: 66 (13.2%)
Progress: 1000/5100 | Matched: 137 (13.7%)
Progress: 1500/5100 | Matched: 171 (11.4%)
Progress: 2000/5100 | Matched: 205 (10.2%)
Progress: 2500/5100 | Matched: 256 (10.2%)
Progress: 3000/5100 | Matched: 322 (10.7%)
Progress: 3500/5100 | Matched: 382 (10.9%)
Progress: 4000/5100 | Matched: 447 (11.2%)
Progress: 4500/5100 | Matched: 676 (15.0%)
Progress: 5000/5100 | Matched: 703 (14.1%)

Final match rate: 705/5100 (13.8%)


In [17]:
# saving to desktop
df_output = df_billboard.drop(columns=['clean_title', 'clean_artist'])

output_path = os.path.expanduser('~/Desktop/audio_features2.csv')
df_output.to_csv(output_path, index=False)

print(f'Saved to Desktop: audio_features.csv')
print(df_output[['year', 'position', 'title', 'artist', 'bpm', 'valence']].head(10).to_string())

Saved to Desktop: audio_features.csv
   year  position                           title               artist      bpm  valence
0  1975         1      Love Will Keep Us Together   Captain & Tennille      NaN      NaN
1  1975         2               Rhinestone Cowboy        Glen Campbell      NaN      NaN
2  1975         3            Philadelphia Freedom           Elton John      NaN      NaN
3  1975         4  Before the Next Teardrop Falls        Freddy Fender   91.041    0.722
4  1975         5              My Eyes Adored You        Frankie Valli      NaN      NaN
5  1975         6          Some Kind of Wonderful  Grand Funk Railroad      NaN      NaN
6  1975         7                    Shining Star   Earth, Wind & Fire  102.587    0.847
7  1975         8                            Fame          David Bowie      NaN      NaN
8  1975         9            Laughter in the Rain          Neil Sedaka      NaN      NaN
9  1975        10             One of These Nights               Eagles  1